## CIND820 Data Preparation

The raw data from QMJHL game centres must undergo numerous calculations and transformations to prepare the variables relevant to this data investigation. The variables for the expected goals (xG) models are as follows:

* Shot distance
* Shot angle
* Time since last shot
* Distance from last shot
* Current shot coordinates
* Previous shot coordinates
* Time elapsed in game
* Player strength
* Score differential

This will be built off the raw API from the QMJHL Game Centre pages.

In [1]:
import pandas as pd

raw = pd.read_csv('2026-27 QMJHL PBP.csv')

There is a large number of columns that can be removed from this dataset given our intended use of the data. Any rows where 'event' = 'goalie_change' will also be deleted, since these data points are not needed because the necessary information is provided in other columns of the remaining dataset.

In [2]:
new = raw[['event', 'period_id', 's', 'team_id', 'x_location', 'y_location', 'home', 'game_goal_id', 'player_team_id', 'goalie_team_id', 'player', 'goalie', 'player_served', 'pp', 'penalty_shot', 'minutes', 'goal_type_name', 'game_id']]

new = new[new["event"] != "goalie_change"]

There are a number logical steps that must be considered when cleaning this data for use.

#### Duplicated goal events
Every goal is entered as two separate events, a 'shot' and a 'goal' (since a goal always comes from a shot). We only want to keep one copy of these events to avoid goals being overrepresented/duplicated and skewing the dataset.

Crucially, various important variables are not populated in the 'goal' event ('player_team_id,' 'goalie_team_id,' 'player,' and 'goalie'). Therefore, we are going to remove the 'goal' events from the initial dataset, then reclassify shots that have a 'game_goal_id' as goals.

In [3]:
new = new[new["event"] != "goal"]

new.loc[new['game_goal_id'].notna(), 'event'] = 'goal'

#### Removing incidental penalties

Sometimes, penalties "cancel out" (if an equal number of players on each team are penalized for the same actions, like roughing/fighting). This dataset labels these types of penalties helpfully, where 'pp' (power play) = 0. We do not need the information for these penalties in our analysis (we are keeping track of penalties to identify how many players are on the ice for each team). If 'pp' = 0, that means that the offending team does not lose a player on the ice. Therefore, we can drop every event = 'penalty' entry where 'pp' = 0.

In [4]:
new = new.drop(new[new['pp'] == 0].index)

#### Score differential

One of the variables that will be used to build the model is score differential. This integer will represent the status of the game's score when the shot occurred, such as when the game is tied (score_diff = 0), the shooting team is down by two (score_diff = -2), or the shooting team is up by three (score_diff = 3). A new column called 'goal_change' will help to calculate this by expressing 'goal' events as positive or negative integers. A goal for the home team will be represented by 1, and a goal for the away team will be represented by -1.

Using the .cumsum() (cumulative sum) function in the 'score_diff' column, calculated from the 'goal_change' values, we can fill the 'score_diff' column with an integer representing the shooting team's game situation. We'll also use the .shift(1, fill_value=0) function to ensure that the shot that becomes the goal illustrates the game situation when the shot was taken (for example, we still want the first goal of the game to read 0 in the 'score_diff' column to show that the game was tied when the shot was taken). The .groupby('game_id') function is also used since our dataset contains multiple games in one csv file.

Finally, np.where() allows us to invert the shot differential for shots taken by the away team. For example, if the home team has two more goals than the away team, 'score_diff' needs to read 2 for shots where 'home' = 1 and -2 for shots where 'home' = 0.

In [5]:
import numpy as np

new['score_diff'] = None

new['goal_change'] = 0

new.loc[(new['event'] == 'goal') & (new['home'] == 1), 'goal_change'] = 1
new.loc[(new['event'] == 'goal') & (new['home'] == 0), 'goal_change'] = -1

new['score_diff'] = new.groupby('game_id')['goal_change'].cumsum().shift(1, fill_value=0)

new['score_diff'] = np.where(new['home'] == 0, -new['score_diff'], new['score_diff'])

#### Time elapsed in game

In the original dataset, the 's' variable (expressing time elapsed in seconds) only expresses the time elapsed in each period, not the entire game. Because we want to analyze the timing of events in the game as a whole (looking at early/late goals, for example), 1200 seconds (20 minutes) will be added to any 's' value where 'period_id' = 2. 2400 seconds will be added to 's' values where 'period_id' = 3, and 3600 seconds will be added to 's' values where 'period_id' = 4.

In [6]:
new['s'] = np.where(new['period_id'] == 2, new['s'] + 1200, new['s'])
new['s'] = np.where(new['period_id'] == 3, new['s'] + 2400, new['s'])
new['s'] = np.where(new['period_id'] == 4, new['s'] + 3600, new['s'])

#### Play strength classification

Penalties have remained in the dataset thus far because they provide crucial information that will be important in the analysis. Because teams generally generate more shots and goals on the power play (5-on-4) because they have more players on the ice than the other team, this is an important variable to consider. Having this data can help the model identify shots and goals that are more likely to go in because the defending team is at a disadvantage.

One preliminary consideration in the data is that double-minor (four-minute-long) penalties are inputted as two penalties of two minutes each. We want to transform these into one four-minute-long penalty entry, especially because if the other team scores during a four-minute power play, their power play continues (as opposed to a two-minute power play, where it ends after the goal).

This is done by making a mask using .duplicated() with a subset of 's', 'game_id', and 'player_served' - finding duplicates where the game, time of the event, and the player ID of who's serving the penalty are identical (double-minor penalties are served to the same player; we need to include this player ID in the subset because sometimes multiple different players will get penalties at the same time). The double_minors DataFrame made from these conditions will adopt a 'minutes' value of 4.0 before being incorporated back into the main new DataFrame. From the new DataFrame, the duplicated events will be dropped (the second copy of the double_minors where 'minutes' = 4.0, as well as the original penalties where 'minutes' = 2.0). These double_minors events are reintegrated into the dataset in their correct original order.

In [7]:
double_minor_mask = new.duplicated(subset=['s', 'game_id', 'player_served'], keep=False)

double_minors = new[double_minor_mask & (new['event'] == 'penalty')]

double_minors['minutes'] = 4.0

new.loc[double_minors.index, :] = double_minors[:]

new = new.drop_duplicates(subset=['s', 'game_id', 'player_served'], keep='first')

/var/folders/0n/31k33nxn59sgvdqjp51nfl2c0000gn/T/ipykernel_2871/3438194613.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  double_minors['minutes'] = 4.0


Below are two functions filling out the new 'home_strength' and 'away_strength' variables to keep track of how many players for each team are on the ice for any given event. We want to keep track of the number of penalties on either team because it affects how many players are on the ice for each team, which affects shots and goals (a power play, having more players on the ice than the other team, often leads to more goals).

This user-defined function takes 'event,' 's,' 'minutes,' and 'home.' into consideration while being able to store active penalties and their start and end times to label the strengh (number of players on the ice) during an event. In calc_home_strength, when event = 'penalty' and home = 1, the 's' value in that row is appended to the active_penalties list, as well as the end time ('s' + 'minutes' * 60, to convert 'minutes' from minutes into seconds). active_count dictates the number that will be printed in the 'home_strength' row, which will be a positive integer before further data transformation.

We also need to be able to end minor penalties if a goal is scored, but not double-minors or major (5 minutes) penalties. In the calc_home_strength function, this loop occurs if event = 'goal' and active_count > 0 and home = 0 (if the away team scores). If there is an active penalty and its duration is 120 seconds (2 minutes), then the end time of the penalty is overwritten as the time of the goal, because this is the situation where a goal for the other team ends the penalty.

In [8]:
new['home_strength'] = np.nan
new['away_strength'] = np.nan

def calc_home_strength(new):
    strengths = []
    active_penalties = []
    
    for idx, row in new.iterrows():
        event = row['event']
        time = row['s']
        duration = row['minutes'] * 60 # converted from minutes (2.0) to seconds (120.0) to match 's' format
        home = row['home']

        active_penalties = [p for p in active_penalties if p[1] > time]
        
        if event == 'penalty' and home == 1:
            active_penalties.append([time, time + duration])
            active_penalties.sort(key=lambda x: x[0])
        
        active_count = sum(1 for start, end in active_penalties if start <= time < end)

        row_strength = active_count
        
        # If a goal is scored, end the oldest active minor penalty. Only one penalty gets ended at a time by a goal (for example, if a team scores at 5v3, one player is released and they then play 5v4).
        if event == 'goal' and active_count > 0 and home == 0:
            for p in active_penalties:
                if p[0] <= time < p[1]  and (p[1] - p[0] == 120.0):
                    p[1] = time
                    break
            active_penalties = [p for p in active_penalties if p[1] > time]
            
        strengths.append(row_strength)
        
    new['home_strength'] = strengths
    return new

new = calc_home_strength(new)

def calc_away_strength(new):
    strengths = []
    active_penalties = []
    
    for idx, row in new.iterrows():
        event = row['event']
        time = row['s']
        duration = row['minutes'] * 60 # converted from minutes (2.0) to seconds (120.0) to match 's' format
        home = row['home']

        active_penalties = [p for p in active_penalties if p[1] > time]
        
        if event == 'penalty' and home == 0:
            active_penalties.append([time, time + duration])
            active_penalties.sort(key=lambda x: x[0])
        
        active_count = sum(1 for start, end in active_penalties if start <= time < end)

        row_strength = active_count
        
        # If a goal is scored, end the oldest active minor penalty. Only one penalty gets ended at a time by a goal (for example, if a team scores at 5v3, one player is released and they then play 5v4).
        if event == 'goal' and active_count > 0 and home == 1:
            for p in active_penalties:
                if p[0] <= time < p[1] and (p[1] - p[0] == 120.0):
                    p[1] = time
                    break
            active_penalties = [p for p in active_penalties if p[1] > time]
            
        strengths.append(row_strength)
        
    new['away_strength'] = strengths
    return new

new = calc_away_strength(new)

After these functions, the outputs are standardized to communicate team strength in terms of how many players each team has on the ice. However, we need to consider a special case when period_id = 4. In overtime, which is regularly played at 3v3 instead of 5v5, if a team has a penalty, the other team will play at a 4v3 advantage (instead of 3v2).

After this step, these numbers can be combined into a object with a format such as '5v5.' This is ordered so that the number of players on the team taking the shot or scoring the goal comes first. We need to use different calculations for regulation time vs. overtime. To avoid complicating the defined function above, this is the step where the home_strength and away_strength will be processed.

Example: If the home team gets a penalty with 60 seconds left in the third period, a shot they take with 30 seconds left in the third period will have its strength calculated as [negative(number of home team penalties) + 5] v [negative(number of away team penalties) + 5] = 4v5. In overtime, this is calculated as [number of away team penalties + 3] v [number of home team penalties + 3] = 3v4. The away team in this scenario gets an extra player because they have a power play. The lines below ensure that overtime strength is expressed accurately.

In [9]:
str_conds = [
    (new['home'] == 1) & (new['period_id'] != 4),
    (new['home'] == 0) & (new['period_id'] != 4),
    (new['home'] == 1) & (new['period_id'] == 4),
    (new['home'] == 0) & (new['period_id'] == 4)
]

str_calcs = [
    (-new['home_strength'].shift(1, fill_value=0).abs() + 5).astype(str) + "v" + (-new['away_strength'].shift(1, fill_value=0).abs() + 5).astype(str),
    (-new['away_strength'].shift(1, fill_value=0).abs() + 5).astype(str) + "v" + (-new['home_strength'].shift(1, fill_value=0).abs() + 5).astype(str),
    (new['away_strength'].shift(1, fill_value=0) + 3).astype(str) + "v" + (new['home_strength'].shift(1, fill_value=0) + 3).astype(str),
    (new['home_strength'].shift(1, fill_value=0) + 3).astype(str) + "v" + (new['away_strength'].shift(1, fill_value=0) + 3).astype(str)
]

new['strength'] = np.select(str_conds, str_calcs, default=np.nan)

#### Empty net goals

We don't want to include empty net shots or goals (when the losing team pulls their goalie for a sixth skater to increase their chances of scoring a goal) in the actual analysis. We still need to wait to delete them until after the score differential and team strength is calculated, as it is possible for a team to score after an empty net goal and we want this to be reflected in the dataset. It is also possible for these goals to happen on the power play, which means the penalty would end with that goal being scored.

A shot on goal is defined as when the goalie has to make a save to prevent the puck from entering the net; therefore every shot recorded on an empty net is a goal. The 'goal_type' and 'goal_type_name' variables for both the 'shot' and 'goal' entry of the event will have 'EN' in these cases. We want to exclude these goals entirely from our analysis. It should not affect our game prediction model, as virtually every empty-net goal is for the team that is already winning.

In [10]:
new = new[~new["goal_type_name"].str.contains("EN", na=False)]

#### Formatting player and goalie names
The 'player' and 'goalie' variables in the raw dataset are fully encoded in JSON format like the example below:

* {'player_id': '23329', 'jersey_number': '19', 'team_id': '8', 'team_code': 'SNB', 'first_name': 'Alexis', 'last_name': 'Joseph'}

We want this example above to read as just 'Alexis Joseph.' .str.extract() can help with that, using regular expression (Regex). In this case, we are extracting and combining what follows "'first_name':" and "'last_name':," with a space in between. The enclosed capturing group, (.*?), captures all of the text in these single quotation marks with the period and the asterisk, then the question mark ensures that it stops after the first match (a comma after the first name, and a bracket after the last name).

Regex documentation/cheat sheet: https://coderpad.io/regular-expression-cheat-sheet/ 

It is not possible with this dataset to unilaterally strip the single quotation mark in this first step, because certain names use double quotations to handle names with apostrophes. Take a look at this real example from the dataset to see how double quotations are sometimes used in these cases:

* {'player_id': '23331', 'jersey_number': '77', 'team_id': '5', 'team_code': 'Hal', 'first_name': 'Malik', 'last_name': **"L'Italien"**}

In [11]:
new["player"] = (
    new["player"].str.extract(r"'first_name': (.*?),")
    + " "
    + new["player"].str.extract(r"'last_name': (.*?)}")
)

new["goalie"] = (
    new["goalie"].str.extract(r"'first_name': (.*?),")
    + " "
    + new["goalie"].str.extract(r"'last_name': (.*?)}")
)

The outputs following this initial step follow the format below:
* 'Alexis' 'Joseph'
* 'Malik' "L'Italien"

After this, we run through both the 'player' and 'goalie' columns again to strip any single or double quotations from the beginning (.str.lstrip()) or end (.str.rstrip()) of these name outputs. Then, to make sure that apostrophes in names are not deleted, .str.replace() is used to circulate through removing single or doubles quotations that are preceded or succeeded by a space. After this, our example outputs will be properly formatted like so:
* Alexis Joseph
* Malik L'Italien

This information is needed in the dataset because the model can be used to evaluate individual performance. In particular, Question 2 will deal in using the models to identify players to trade for.

In [12]:
# Strip quotations at beginning
new['player'] = new['player'].str.lstrip('\'"')
new['goalie'] = new['goalie'].str.lstrip('\'"')
# Strip quotations at end
new['player'] = new['player'].str.rstrip('\'"')
new['goalie'] = new['goalie'].str.rstrip('\'"')
# Strip quotations in middle
new['player'] = new['player'].str.replace("' ", " ")
new['player'] = new['player'].str.replace(" '", " ")
new['goalie'] = new['goalie'].str.replace("' ", " ")
new['goalie'] = new['goalie'].str.replace(" '", " ")
new['player'] = new['player'].str.replace('" ', " ")
new['player'] = new['player'].str.replace(' "', " ")
new['goalie'] = new['goalie'].str.replace('" ', " ")
new['goalie'] = new['goalie'].str.replace(' "', " ")

#### Replacing team IDs with team names

The 'team_id,' 'player_team_id,' and 'goalie_team_id' variables all take integers to encode team names more simply. Below lists the team IDs as assigned by the API, matched with their team names.

* 1 = Moncton Wildcats
* 2 = Newfoundland Regiment
* 3 = Cape Breton Eagles
* 5 = Halifax Mooseheads
* 7 = Charlottetown Islanders
* 8 = Saint John Sea Dogs
* 9 = Quebec Remparts
* 10 = Chicoutimi Sagueneens
* 11 = Rouyn-Noranda Huskies
* 12 = Gatineau Olympiques
* 13 = Shawinigan Cataractes
* 14 = Drummondville Voltigeurs
* 15 = Val-d'Or Foreurs
* 16 = Baie-Comeau Drakkar
* 17 = Victoriaville Tigres
* 18 = Rimouski Oceanic
* 19 = Blainville-Boisbriand Armada
* 60 = Sherbrooke Phoenix

This information is needed in the dataset to be able to evaluate teams' performance within the models, which will be used to create the win prediction model.

In [13]:
teamdict = {
    1: "Moncton",
    2: "Newfoundland",
    3: "Cape Breton",
    5: "Halifax",
    7: "Charlottetown",
    8: "Saint John",
    9: "Quebec",
    10: "Chicoutimi",
    11: "Rouyn-Noranda",
    12: "Gatineau",
    13: "Shawinigan",
    14: "Drummondville",
    15: "Val-d'Or",
    16: "Baie-Comeau",
    17: "Victoriaville",
    18: "Rimouski",
    19: "Blainville-Boisbriand",
    60: "Sherbrooke"
}

new["team_id"] = new["team_id"].replace(teamdict)
new["player_team_id"] = new["player_team_id"].replace(teamdict)
new["goalie_team_id"] = new["goalie_team_id"].replace(teamdict)

#### Removing data no longer needed

Before moving onto the final calculated fields, we can simplify the dataset by removing some of the rows and columns previously needed to calculate the calculated fields above.

There is one penalty shot goal in the dataset, which doesn't have its coordinates tracked. Because penalty shots and shootouts are unique 1v0 shooting situations, we will drop this from the dataset. Younggren & Younggren similarly excluded penalty shots and penalty shot goals from their NHL regression model. The score status remains encoded.

Penalties no longer need to be included because we have already calculated the strength (number of players on the ice) for each shot event.

In [14]:
new = new[new['goal_type_name'] != 'PS']
new = new.drop(columns=['team_id', 'game_goal_id', 'player_served', 'pp', 'penalty_shot', 'minutes', 'goal_type_name', 'goal_change', 'home_strength', 'away_strength'])
new = new[new['event'] != 'penalty']
new = new[new['event'] != 'penaltyshot']

#### Time since previous shot

One of the key variables featured in NHL-level analysis is time since previous shot. This is because 'flurries' of shots that are close together are considered dangerous chances with a higher likelihood of a goal. Even sustained pressure of 5 to 20 seconds between shots can increase the chance of a goal, as well.

When we calculate this, we want to make sure that times from previous games and periods will not be included in the analysis, therefore we will use the .groupby() function to calculate each game and period separate of each other. Now that we just removed penalty events from the dataset, the dataset now only contains shots and goals. Therefore, we don't need to filter out penalties.

In [15]:
new['s_since_prev'] = new.groupby(['game_id', 'period_id'])['s'].diff()

#### Converting coordinates to ft

The QMJHL API uses an arbitrary measurement of 600 units by 300 units for the ice surface. We want to fit this to the North American standard ice surface size of 200 ft to 85 ft. This step must be taken here to accurately calculate the rest of the fields, which relate to shot locations.

In [16]:
new['x_location'] = new['x_location'] / 3 # (600 / 200 = 3)
new['y_location'] = new['y_location'] / (300 / 85)

#### Previous shot coordinates

We want to carry down the x and y coordinates of the previous shot because this is another factor that can affect the likelihood of a goal. Just like how the coordinates of the current shot can affect the likelihood of a goal, this dimension could bring some clarity to shots that are close together in time, as mentioned just above.

In [17]:
new['x_prev'] = new.groupby(['game_id', 'period_id'])['x_location'].shift(1)
new['y_prev'] = new.groupby(['game_id', 'period_id'])['y_location'].shift(1)

#### Distance from previous shot

The fourth-strongest variable in Younggren & Younggren's 2021 analysis on NHL data, the previous shot coordinates just populated can also be used to measure the distance between two shooting locations. This could have a strong impact on this model as well, as a large value in this variable alongside a small value in the 's_since_prev' variable creates typically dangerous scoring chances. This metric will be measured using the Euclidean distance formula.

In [18]:
new['distance_prev'] = np.sqrt((new['x_location'] - new['x_prev']) ** 2 + (new['y_location'] - new['y_prev']) ** 2)

#### Shot location standardization

We have adjusted 'x_location' and 'y_location' to reflect the ice surface's real dimensions as 200 ft by 85 ft. Upon further investigation, the data shows that the away team (where 'home' = 0)'s attacking zone is found at x_location = 100 to 200. It is important to normalize these results relative to each other for direct comparison between home and away games for every team. This is done by subtracting the x_location from 200 and subtracting the y_location from 85 whenever 'home' = 0.

In [19]:
new['x_location'] = np.where((new['home'] == 0), 200 - new['x_location'], new['x_location'])
new['y_location'] = np.where((new['home'] == 0), 85 - new['y_location'], new['y_location'])

We also need to manage the x_prev and y_prev variables. When the previous shot was from the same team, the values do not need to change. However, when the previous shot was from the other team, the coordinate values should change to reflect the distance between shots accurately.

In [20]:
new['x_prev'] = np.where(new['home'] == new['home'].shift(1), new['x_prev'], 200 - new['x_prev'])
new['y_prev'] = np.where(new['home'] == new['home'].shift(1), new['y_prev'], 85 - new['y_prev'])

#### Shot distance

As mentioned in the Interim Report, the net's location on the 600 unit x 300 unit plane used in the API to track shot locations is (33, 150). Converted to ft, this is (11, 42.5). The Euclidean distance formula will be used to calculate shot distance from the net, using the dataset's arbitrary unit of measurement.

In [21]:
net_location = (11, 42.5)

new['shot_distance'] = np.sqrt((new['x_location'] - net_location[0]) ** 2 + (new['y_location'] - net_location[1]) ** 2)

#### Shot angle

The .arctan2 and .degrees mathematical functions in Numpy will be used to calculate the angle between the location coordinates of the shot and the net's location. The values are also standardized as absolute values. When shot_angle = 0, this means that the shot was taken directly in front of the net, which is generally considered the most dangerous shooting position.

In [22]:
new['shot_angle'] = (np.degrees(np.arctan2((new['y_location'] - net_location[1]), (new['x_location'] - net_location[0])))).abs()

In [23]:
new = new.reset_index(drop=True)

In [24]:
new.to_csv('2026-27 QMJHL PBP prepared.csv', index=False)